# 07 – NewsAPI, Assistent und Export

**Projekt:** WealthScope AI 1.0
**Methode:** QUA³CK · reproduzierbarer Out-of-Time-Benchmark
**Hinweis:** Wissenschaftlicher Prototyp, keine Anlageberatung.

## Zusatzmodule mit klaren Grenzen

Diese Module erhöhen Aktualität und Verständlichkeit, verändern aber nicht
rückwirkend den historischen ML-Benchmark.

## Lernziele

Nach diesem Notebook könnt ihr:

- generative KI von prädiktivem ML sauber trennen
- Leitplanken für einen erklärenden Assistenten formulieren
- einen Export so gestalten, dass er ohne die App interpretierbar bleibt

### Warum die Trennung nicht kosmetisch ist

Im Projekt kommen zwei völlig verschiedene Dinge vor, die beide „KI" heißen:

| | Random Forest | Gemini-Assistent |
|---|---|---|
| Aufgabe | Wahrscheinlichkeit schätzen | Kontext in Sprache fassen |
| Trainiert auf | 120.920 Kursbeobachtungen bis 2006 | fremdem Textkorpus |
| Prüfbar durch | Out-of-Time-Test, ROC-AUC | keine projektinterne Metrik |
| Fehlermodus | schwaches Signal | plausibel klingende Falschaussage |
| Im Ergebnis der Arbeit | ja, er *ist* das Ergebnis | nein, reine Bedienhilfe |

Würde der Assistent Prognosen formulieren, wären sie durch **nichts** in diesem
Projekt gedeckt — die gesamte Validierungsarbeit bezieht sich ausschließlich auf
den Random Forest. Die Trennung ist deshalb eine methodische Notwendigkeit.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "wealthscope_features.parquet"
DIAGNOSTICS_PATH = PROJECT_ROOT / "models" / "diagnostics.json"
EXPERIMENTS_PATH = PROJECT_ROOT / "models" / "validation_experiments.json"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Datensatz fehlt: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(f"Daten: {len(df):,} Zeilen × {len(df.columns)} Spalten")
display(df.head(3))

Daten: 192,119 Zeilen × 27 Spalten


,date,open,high,low,close,volume,ticker,asset_type,source_file,daily_return,...,ma_200_distance,volatility_20d,rolling_high,drawdown,future_return_20d,target_20d,open_interest,volatility_60d,rolling_low_60,rolling_high_60
0,1985-06-21,0.25742,0.26381,0.25742,0.25742,46333854,AAPL,Stock,Stocks/aapl.us.txt,0.020374,...,-0.331288,0.040521,0.48791,-0.472403,0.044635,1.0,NaN,NaN,NaN,NaN
1,1985-06-24,0.27535,0.27920,0.27535,0.27535,57384755,AAPL,Stock,Stocks/aapl.us.txt,0.069653,...,-0.283328,0.040440,0.48791,-0.435654,-0.041910,0.0,NaN,NaN,NaN,NaN
2,1985-06-25,0.27920,0.28556,0.27920,0.27920,81966629,AAPL,Stock,Stocks/aapl.us.txt,0.013982,...,-0.271961,0.037120,0.48791,-0.427763,-0.073424,0.0,NaN,NaN,NaN,NaN


In [2]:
modules = pd.DataFrame([
    ["NewsAPI", "externe Schlagzeilen", "Netzwerk/API-Key, Abdeckung, Aktualität"],
    ["Regelbasiertes Sentiment", "transparente Einordnung", "keine tiefe Semantik"],
    ["KI-Assistent", "Erklärung der aktuellen Ansicht", "Halluzination, keine Beratung"],
    ["Export", "CSV, JSON, Markdown, ZIP", "Zeitstempel und Annahmen mitliefern"],
], columns=["Modul", "Nutzen", "Grenze"])
modules

,Modul,Nutzen,Grenze
0,NewsAPI,externe Schlagzeilen,"Netzwerk/API-Key, Abdeckung, Aktualität"
1,Regelbasiertes Sentiment,transparente Einordnung,keine tiefe Semantik
2,KI-Assistent,Erklärung der aktuellen Ansicht,"Halluzination, keine Beratung"
3,Export,"CSV, JSON, Markdown, ZIP",Zeitstempel und Annahmen mitliefern


In [3]:
assistant_guardrails = [
    "Keine Kauf-, Verkaufs- oder Renditeversprechen.",
    "Modellkennzahlen und Datenzeitraum nennen.",
    "Historische Signale nicht als Kausalität darstellen.",
    "Bei fehlenden Live-Daten den Zustand transparent ausweisen.",
    "Export muss Annahmen, Quelle und Zeitstempel enthalten.",
]
pd.DataFrame({"Leitplanke": assistant_guardrails})

,Leitplanke
0,"Keine Kauf-, Verkaufs- oder Renditeversprechen."
1,Modellkennzahlen und Datenzeitraum nennen.
2,Historische Signale nicht als Kausalität darst...
3,Bei fehlenden Live-Daten den Zustand transpare...
4,"Export muss Annahmen, Quelle und Zeitstempel e..."


In [4]:
export_contract = {
    "app_version": "1.0.5",
    "model_metrics_source": "models/diagnostics.json",
    "method": "purged out-of-time + expanding walk-forward",
    "target": "target_20d",
    "financial_advice": False,
}
print(json.dumps(export_contract, ensure_ascii=False, indent=2))

{
  "app_version": "1.0.5",
  "model_metrics_source": "models/diagnostics.json",
  "method": "purged out-of-time + expanding walk-forward",
  "target": "target_20d",
  "financial_advice": false
}


## Schluss

Ein gutes KI-Produkt macht Unsicherheit sichtbar. News, Assistent und Export
sind daher Kommunikationsschichten – keine Abkürzung zu einer stärkeren
Prognose.

## Management-Checkpoint

Der Assistent darf rechnen und formulieren; verantworten muss die Aussage der
Mensch, der sie fachlich begründen kann. Genau deshalb nennt der Export-Vertrag
oben `financial_advice: False` und verweist auf `models/diagnostics.json` als
Quelle der Kennzahlen: Ein exportierter Bericht muss auch dann noch korrekt
interpretierbar sein, wenn niemand mehr weiß, in welcher App er entstanden ist.